# Notebook 7: Integration — Export to YakOS App

**Goal:** Wire the trained models into the YakOS Streamlit app.

## Deliverables

### New `yak_core/projections.py` functions (already implemented)
- `yakos_fp_projection(player_features)` → `{proj, floor, ceil}`
- `yakos_minutes_projection(player_features)` → `{proj_minutes}`
- `yakos_ownership_projection(player_features)` → `{proj_own}`
- `yakos_ensemble(yakos_proj, tank01_proj, rg_proj, weights)` → blended float

### Updated `fetch_live_opt_pool` (already implemented)
- Stores `tank01_proj` alongside `proj` as the original Tank01 projection
- Adds `proj_source` column

### Actuals bug fix (already implemented)
- `fetch_actuals_from_api` now always uses box scores (never DFS projections as actuals)

This notebook demonstrates how to apply YakOS projection functions to a live pool.

In [ ]:
# Make sure yak_core is on the path
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
from yak_core.projections import (
    yakos_fp_projection,
    yakos_minutes_projection,
    yakos_ownership_projection,
    yakos_ensemble,
)

print('YakOS projection functions imported successfully.')

## Enrich a Live Pool with YakOS Projections

After fetching a live pool from `fetch_live_opt_pool`, apply YakOS projection functions to add `yakos_proj`, `yakos_proj_minutes`, `yakos_proj_own`, and `proj` (the final ensemble blend).

In [ ]:
def apply_yakos_projections(pool_df: pd.DataFrame,
                             ensemble_weights: dict = None) -> pd.DataFrame:
    """
    Apply all YakOS projection functions to a live pool DataFrame.

    Expected columns (any subset): salary, tank01_proj, rg_proj,
    rolling_fp_5, rolling_fp_10, rolling_fp_20, rolling_min_5,
    rolling_min_10, b2b, spread, rg_ownership.

    Adds columns: yakos_proj, yakos_floor, yakos_ceil,
                  yakos_proj_minutes, yakos_proj_own,
                  proj (final ensemble), proj_source.
    """
    if ensemble_weights is None:
        ensemble_weights = {'yakos': 0.40, 'tank01': 0.30, 'rg': 0.30}

    df = pool_df.copy()

    # Build per-player feature dicts and call projection functions
    fp_projs, fp_floors, fp_ceils = [], [], []
    min_projs, own_projs = [], []

    for _, row in df.iterrows():
        feat = row.to_dict()

        fp_res  = yakos_fp_projection(feat)
        min_res = yakos_minutes_projection(feat)
        own_res = yakos_ownership_projection(feat)

        fp_projs.append(fp_res['proj'])
        fp_floors.append(fp_res['floor'])
        fp_ceils.append(fp_res['ceil'])
        min_projs.append(min_res['proj_minutes'])
        own_projs.append(own_res['proj_own'])

    df['yakos_proj']         = fp_projs
    df['yakos_floor']        = fp_floors
    df['yakos_ceil']         = fp_ceils
    df['yakos_proj_minutes'] = min_projs
    df['yakos_proj_own']     = own_projs

    # Preserve original sources
    if 'tank01_proj' not in df.columns:
        df['tank01_proj'] = df.get('proj', np.nan)
    rg_proj_col = df.get('rg_proj', pd.Series(np.nan, index=df.index))

    # Build final ensemble projection
    df['proj'] = [
        yakos_ensemble(
            yakos_proj=df['yakos_proj'].iloc[i],
            tank01_proj=df['tank01_proj'].iloc[i],
            rg_proj=rg_proj_col.iloc[i] if hasattr(rg_proj_col, 'iloc') else np.nan,
            weights=ensemble_weights,
        )
        for i in range(len(df))
    ]
    df['proj_source'] = 'yakos_ensemble'

    return df


# ----- Demo with sample pool -----
sample_pool = pd.DataFrame([
    {'player_name': 'LeBron James',    'team': 'LAL', 'pos': 'SF', 'salary': 10800, 'tank01_proj': 54.0, 'rolling_fp_5': 55.0, 'rolling_min_5': 35.0},
    {'player_name': 'Stephen Curry',   'team': 'GSW', 'pos': 'PG', 'salary': 10200, 'tank01_proj': 50.0, 'rolling_fp_5': 49.0, 'rolling_min_5': 34.0},
    {'player_name': 'Value Play A',    'team': 'BOS', 'pos': 'PG', 'salary': 4500,  'tank01_proj': 18.0, 'rolling_fp_5': 20.0, 'rolling_min_5': 22.0},
    {'player_name': 'Value Play B',    'team': 'BOS', 'pos': 'SG', 'salary': 5200,  'tank01_proj': 22.0, 'rolling_fp_5': 24.0, 'rolling_min_5': 26.0},
    {'player_name': 'B2B Risk Player', 'team': 'MIA', 'pos': 'PF', 'salary': 7800,  'tank01_proj': 38.0, 'rolling_fp_5': 36.0, 'rolling_min_5': 32.0, 'b2b': True},
])

enriched = apply_yakos_projections(sample_pool)
display_cols = ['player_name', 'salary', 'tank01_proj', 'yakos_proj', 'yakos_floor', 'yakos_ceil',
                'yakos_proj_minutes', 'yakos_proj_own', 'proj', 'proj_source']
print(enriched[[c for c in display_cols if c in enriched.columns]].to_string())

## Integration with `fetch_live_opt_pool`

The updated `fetch_live_opt_pool` in `yak_core/live.py` now returns:
- `tank01_proj` — Tank01's original projection (preserved)
- `proj_source = 'tank01'` — tracks where the projection came from

After fetching, you can call `apply_yakos_projections()` to enrich with YakOS models.

In [ ]:
# Example integration (requires RAPIDAPI_KEY)
RAPIDAPI_KEY = os.environ.get('RAPIDAPI_KEY', '')

if RAPIDAPI_KEY:
    from yak_core.live import fetch_live_opt_pool
    cfg = {'RAPIDAPI_KEY': RAPIDAPI_KEY}
    slate_date = pd.Timestamp.today().strftime('%Y-%m-%d')

    try:
        live_pool = fetch_live_opt_pool(slate_date, cfg)
        print(f'Live pool: {len(live_pool)} players')
        print(f'Columns: {list(live_pool.columns)}')

        # Enrich with YakOS projections
        enriched_live = apply_yakos_projections(live_pool)
        print('\nTop 10 by YakOS ensemble proj:')
        print(
            enriched_live.nlargest(10, 'proj')[
                ['player_name', 'team', 'pos', 'salary', 'tank01_proj', 'yakos_proj', 'proj']
            ].to_string()
        )
    except Exception as e:
        print(f'Could not fetch live pool: {e}')
else:
    print('RAPIDAPI_KEY not set. Skipping live pool fetch.')

## Summary of Changes Made

| File | Change |
|------|--------|
| `yak_core/projections.py` | Added `yakos_fp_projection`, `yakos_minutes_projection`, `yakos_ownership_projection`, `yakos_ensemble` |
| `yak_core/live.py` | Fixed `fetch_actuals_from_api` — always uses box scores, never DFS endpoint |
| `yak_core/live.py` | `fetch_live_opt_pool` now adds `tank01_proj` and `proj_source` columns |
| `tests/test_live_actuals.py` | Updated tests to verify new box-score-only actuals behavior |
| `tests/test_projections.py` | Added 27 tests for the four new projection functions |

## Next Steps After Model Training

1. Run Notebooks 1→2→3→4→5 to build and train models
2. Copy `models/yakos_fp_model.pkl`, `yakos_minutes_model.pkl`, `yakos_ownership_model.pkl` to the repo root (or set `YAKOS_ROOT`)
3. The `yakos_fp_projection` / `yakos_minutes_projection` / `yakos_ownership_projection` functions auto-load the pickle if present
4. Call `apply_yakos_projections(pool_df)` anywhere in the app to get full YakOS enrichment